<a href="https://colab.research.google.com/github/ANSH76564/banking_/blob/main/banking_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sqlite3
import sys

# Database Configuration
DB_FILE = "bank_system.db"


# ==========================================
# DATABASE SETUP & INITIALIZATION
# ==========================================


def get_connection():
    """Returns a connection to the SQLite database."""
    return sqlite3.connect(DB_FILE)


def init_db():
    """Create database tables and track overall bank financials."""
    conn = get_connection()
    cursor = conn.cursor()

    # Create Customers Table
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS customers (
            customer_id TEXT PRIMARY KEY,
            name TEXT NOT NULL,
            phone TEXT NOT NULL,
            balance REAL DEFAULT 0.0,
            loan REAL DEFAULT 0.0
        )
    """
    )

    # Create Transactions Table
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS transactions (
            trans_id INTEGER PRIMARY KEY AUTOINCREMENT,
            customer_id TEXT NOT NULL,
            type TEXT NOT NULL,
            amount REAL NOT NULL,
            description TEXT NOT NULL,
            FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
        )
    """
    )

    # Create Bank Financials Table
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS bank_ledger (
            id INTEGER PRIMARY KEY CHECK (id = 1),
            expenses REAL DEFAULT 0.0,
            income REAL DEFAULT 0.0
        )
    """
    )

    # Initialize single ledger row if missing
    cursor.execute(
        "INSERT OR IGNORE INTO bank_ledger (id, expenses, income) VALUES (1, 0.0, 0.0)"
    )

    conn.commit()
    conn.close()


# ==========================================
# BANKING FUNCTIONS
# ==========================================


def create_customer():
    print("\n--- CREATE CUSTOMER ---")
    customer_id = input("Enter Customer ID: ").strip()

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT customer_id FROM customers WHERE customer_id = ?",
        (customer_id,),
    )
    if cursor.fetchone():
        print("Customer already exists!")
        conn.close()
        return

    name = input("Enter customer name: ").strip()
    phone = input("Enter phone number: ").strip()

    cursor.execute(
        "INSERT INTO customers (customer_id, name, phone, balance, loan) VALUES (?, ?, ?, 0.0, 0.0)",
        (customer_id, name, phone),
    )
    conn.commit()
    conn.close()
    print("Customer created successfully!")


def view_customer():
    customer_id = input("Enter Customer ID: ").strip()

    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT customer_id, name, phone, balance, loan FROM customers WHERE customer_id = ?",
        (customer_id,),
    )
    row = cursor.fetchone()
    conn.close()

    if not row:
        print("Customer not found!")
        return

    print("\n--- CUSTOMER DETAILS ---")
    print("Customer ID:", row[0])
    print("Name:", row[1])
    print("Phone:", row[2])
    print("Balance:", row[3])
    print("Loan:", row[4])


def deposit():
    customer_id = input("Enter Customer ID: ").strip()
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT balance FROM customers WHERE customer_id = ?", (customer_id,)
    )
    row = cursor.fetchone()

    if not row:
        print("Customer not found!")
        conn.close()
        return

    try:
        amount = float(input("Enter deposit amount: "))
        if amount <= 0:
            print("Invalid amount!")
            conn.close()
            return
    except ValueError:
        print("Invalid numerical input!")
        conn.close()
        return

    # Update balance and log transaction
    cursor.execute(
        "UPDATE customers SET balance = balance + ? WHERE customer_id = ?",
        (amount, customer_id),
    )
    cursor.execute(
        "INSERT INTO transactions (customer_id, type, amount, description) VALUES (?, 'Deposit', ?, ?)",
        (customer_id, amount, f"Deposited: {amount}"),
    )

    conn.commit()
    conn.close()
    print("Money deposited successfully!")


def withdraw():
    customer_id = input("Enter Customer ID: ").strip()
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT balance FROM customers WHERE customer_id = ?", (customer_id,)
    )
    row = cursor.fetchone()

    if not row:
        print("Customer not found!")
        conn.close()
        return

    current_balance = row[0]

    try:
        amount = float(input("Enter withdrawal amount: "))
        if amount <= 0:
            print("Invalid amount!")
            conn.close()
            return
        elif amount > current_balance:
            print("Insufficient balance!")
            conn.close()
            return
    except ValueError:
        print("Invalid numerical input!")
        conn.close()
        return

    # Update balance and log transaction
    cursor.execute(
        "UPDATE customers SET balance = balance - ? WHERE customer_id = ?",
        (amount, customer_id),
    )
    cursor.execute(
        "INSERT INTO transactions (customer_id, type, amount, description) VALUES (?, 'Withdrawal', ?, ?)",
        (customer_id, amount, f"Withdrawn: {amount}"),
    )

    conn.commit()
    conn.close()
    print("Money withdrawn successfully!")


def give_loan():
    customer_id = input("Enter Customer ID: ").strip()
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT loan FROM customers WHERE customer_id = ?", (customer_id,)
    )
    row = cursor.fetchone()

    if not row:
        print("Customer not found!")
        conn.close()
        return

    try:
        amount = float(input("Enter loan amount: "))
        if amount <= 0:
            print("Invalid amount!")
            conn.close()
            return
    except ValueError:
        print("Invalid numerical input!")
        conn.close()
        return

    # Update loan and log transaction
    cursor.execute(
        "UPDATE customers SET loan = loan + ? WHERE customer_id = ?",
        (amount, customer_id),
    )
    cursor.execute(
        "INSERT INTO transactions (customer_id, type, amount, description) VALUES (?, 'Loan Received', ?, ?)",
        (customer_id, amount, f"Loan received: {amount}"),
    )

    conn.commit()
    conn.close()
    print("Loan approved successfully!")


def pay_loan():
    customer_id = input("Enter Customer ID: ").strip()
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT loan FROM customers WHERE customer_id = ?", (customer_id,)
    )
    row = cursor.fetchone()

    if not row:
        print("Customer not found!")
        conn.close()
        return

    current_loan = row[0]

    try:
        amount = float(input("Enter loan payment amount: "))
        if amount <= 0:
            print("Invalid amount!")
            conn.close()
            return
        elif amount > current_loan:
            print("Amount is greater than loan!")
            conn.close()
            return
    except ValueError:
        print("Invalid numerical input!")
        conn.close()
        return

    # Update customer loan, update bank income, log transaction
    cursor.execute(
        "UPDATE customers SET loan = loan - ? WHERE customer_id = ?",
        (amount, customer_id),
    )
    cursor.execute(
        "UPDATE bank_ledger SET income = income + ? WHERE id = 1", (amount,)
    )
    cursor.execute(
        "INSERT INTO transactions (customer_id, type, amount, description) VALUES (?, 'Loan Payment', ?, ?)",
        (customer_id, amount, f"Loan payment: {amount}"),
    )

    conn.commit()
    conn.close()
    print("Loan payment successful!")


def add_expense():
    try:
        amount = float(input("Enter expense amount: "))
        if amount <= 0:
            print("Invalid amount!")
            return
    except ValueError:
        print("Invalid numerical input!")
        return

    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE bank_ledger SET expenses = expenses + ? WHERE id = 1",
        (amount,),
    )
    conn.commit()
    conn.close()
    print("Expense added!")


def bank_financial_report():
    conn = get_connection()
    cursor = conn.cursor()

    # Aggregate total deposits and total loans across all customers
    cursor.execute(
        "SELECT COALESCE(SUM(balance), 0.0), COALESCE(SUM(loan), 0.0) FROM customers"
    )
    total_deposits, total_loans = cursor.fetchone()

    # Fetch bank ledger details
    cursor.execute("SELECT income, expenses FROM bank_ledger WHERE id = 1")
    income, expenses = cursor.fetchone()

    conn.close()

    profit = income - expenses
    print("\n========== BANK FINANCIAL REPORT ==========")
    print("Total Customer Deposits:", total_deposits)
    print("Total Outstanding Loans:", total_loans)
    print("Bank Income:", income)
    print("Bank Expenses:", expenses)
    print("Bank Profit/Loss:", profit)


def transaction_history():
    customer_id = input("Enter Customer ID: ").strip()
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT customer_id FROM customers WHERE customer_id = ?",
        (customer_id,),
    )
    if not cursor.fetchone():
        print("Customer not found!")
        conn.close()
        return

    cursor.execute(
        "SELECT description FROM transactions WHERE customer_id = ? ORDER BY trans_id ASC",
        (customer_id,),
    )
    history = cursor.fetchall()
    conn.close()

    print("\n--- TRANSACTION HISTORY ---")
    if not history:
        print("No transactions found!")
    else:
        for row in history:
            print(row[0])


def view_all_customers():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT customer_id, name, balance, loan FROM customers")
    records = cursor.fetchall()
    conn.close()

    if not records:
        print("No customers found!")
        return

    print("\n========== ALL CUSTOMERS ==========")
    for customer_id, name, balance, loan in records:
        print("\nCustomer ID:", customer_id)
        print("Name:", name)
        print("Balance:", balance)
        print("Loan:", loan)


def bank_management_system():
    # Initialize SQL Database structure on startup
    init_db()

    while True:
        print("\n=================================")
        print("     BANK MANAGEMENT SYSTEM")
        print("=================================")
        print("1. Create Customer")
        print("2. View Customer")
        print("3. Deposit Money")
        print("4. Withdraw Money")
        print("5. Give Loan")
        print("6. Pay Loan")
        print("7. Transaction History")
        print("8. View All Customers")
        print("9. Add Bank Expense")
        print("10. Bank Financial Report")
        print("11. Exit")

        choice = input("\nEnter your choice: ")

        if choice == "1":
            create_customer()
        elif choice == "2":
            view_customer()
        elif choice == "3":
            deposit()
        elif choice == "4":
            withdraw()
        elif choice == "5":
            give_loan()
        elif choice == "6":
            pay_loan()
        elif choice == "7":
            transaction_history()
        elif choice == "8":
            view_all_customers()
        elif choice == "9":
            add_expense()
        elif choice == "10":
            bank_financial_report()
        elif choice == "11":
            print("Thank you for using the Bank Management System!")
            break
        else:
            print("Invalid choice!")


if __name__ == "__main__":
    bank_management_system()


     BANK MANAGEMENT SYSTEM
1. Create Customer
2. View Customer
3. Deposit Money
4. Withdraw Money
5. Give Loan
6. Pay Loan
7. Transaction History
8. View All Customers
9. Add Bank Expense
10. Bank Financial Report
11. Exit
